In [189]:
import pandas as pd
import warnings 
warnings.filterwarnings('ignore')

In [190]:
df = pd.read_json(r"D:\IMP  ML  PROJECTS\CAR PRICE PREDICTION\web scraping\car_dataset_ahmedabad.json")

col_list = df.columns
col_list

Index(['url', 'car_name', 'Price', 'Registration Year', 'Insurance',
       'Fuel Type', 'Seats', 'Kms Driven', 'RTO', 'Ownership',
       ...
       'Secondary Fuel Type', 'Drag Coefficient',
       'Boot Space Rear Seat Folding', 'Approach Angle', 'Break-over Angle',
       'Departure Angle', 'Petrol Mileage (ARAI)',
       'Petrol Fuel Tank Capacity (Litres)', 'Acceleration 0-100kmph',
       'CNG Highway Mileage'],
      dtype='object', length=113)

In [191]:
# with open(r'D:\IMP  ML  PROJECTS\CAR PRICE PREDICTION\web scraping\feature_engineering\features.txt', 'w') as f:
#     for item in col_list:
#         f.write(item + '\n')

In [192]:
duplicate_count = df.duplicated().sum()

In [193]:
print(df.shape)
print(duplicate_count)

(1074, 113)
0


In [194]:
req_col = ['Mileage', 'Engine', 'Kerb Weight', 'Fuel', 'Transmission Type', 'Power', 'No. of Cylinders', 'Registration Year']
df = df[req_col]

# Save df as csv file (dataset.csv)
df.to_csv('dataset.csv', index=False)

In [195]:
df.head()

,Mileage,Engine,Kerb Weight,Fuel,Transmission Type,Power,No. of Cylinders,Registration Year
0,18.9 kmpl,1197 cc,935 kg,Petrol,Manual,82 bhp,4.0,2015
1,19.81 kmpl,1086 cc,860 kg,Petrol,Manual,68.05 bhp,4.0,Apr 2015
2,15.6 kmpl,1196 cc,1090 kg,Petrol,Manual,70 bhp,4.0,Dec 2019
3,18.9 kmpl,1197 cc,1060 kg,Petrol,Manual,81.86 bhp,4.0,Jul 2017
4,25.44 kmpl,936 cc,1025 kg,Diesel,Manual,56.3 bhp,3.0,2015


In [196]:
# Missing percentage count
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100
    print(f"{col}: {missing_percentage:.2f}% missing")

Mileage: 5.03% missing
Engine: 1.30% missing
Kerb Weight: 7.82% missing
Fuel: 11.55% missing
Transmission Type: 0.28% missing
Power: 2.89% missing
No. of Cylinders: 0.84% missing
Registration Year: 0.28% missing


In [197]:
df['Fuel'].value_counts()

Fuel
Petrol    761
Diesel    161
CNG        28
Name: count, dtype: int64

1. Mileage

In [198]:
df['Mileage'] = df['Mileage'].str.extract(r'(\d+\.?\d*)').astype(float)

2. Engine

In [199]:
df['Engine'] = df['Engine'].str.extract(r'(\d+)').astype(float)

3. Weight

In [200]:
df['Kerb Weight'] = df['Kerb Weight'].str.extract(r'(\d+)').astype(float)

4. Power

In [201]:
df['Power'] = df['Power'].str.extract(r'(\d+\.?\d*)').astype(float)

5. Registration Year

In [202]:
df['Registration Year'] = pd.to_numeric(
    df['Registration Year'].str.extract(r'(\d{4})')[0],
    errors='coerce'
)

6. Transmission Type

In [203]:
df['Transmission Type'] = df['Transmission Type'].map({
                                'Automatic': 1,
                                'Manual': 0
                            })

7. Fuel

In [204]:
def encode_and_show_dropped(df, column, drop_first=True):
    dummies = pd.get_dummies(df[column], prefix=column, drop_first=drop_first)
    
    all_categories = df[column].unique()
    encoded_categories = [col.replace(f"{column}_", "") for col in dummies.columns]
    dropped_categories = [cat for cat in all_categories if cat not in encoded_categories]
    
    print(f"All categories:     {list(all_categories)}")
    print(f"Encoded categories: {encoded_categories}")
    print(f"Dropped categories: {dropped_categories}")
    
    return dummies

# usage
fuel_dummies = encode_and_show_dropped(df, 'Fuel', drop_first=True)
df = pd.concat([df.drop(columns=['Fuel']), fuel_dummies], axis=1)


All categories:     ['Petrol', 'Diesel', None, 'CNG']
Encoded categories: ['Diesel', 'Petrol']
Dropped categories: [None, 'CNG']


In [205]:
df['Mileage'].fillna(df['Mileage'].median(), inplace=True)

In [206]:
df.columns

Index(['Mileage', 'Engine', 'Kerb Weight', 'Transmission Type', 'Power',
       'No. of Cylinders', 'Registration Year', 'Fuel_Diesel', 'Fuel_Petrol'],
      dtype='object')

---

In [207]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [208]:
# Features and target
X = df.drop(columns=['Mileage'])
y = df['Mileage']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [209]:
model = RandomForestRegressor()

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],       # ✅ removed 'model__' prefix
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

# Grid search
grid_search = GridSearchCV(
    model,                            # ✅ directly pass model, no pipeline
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print("R2 Score:", r2)

Best parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best CV score: 0.7790575649013505
R2 Score: 0.8623679387214376


In [210]:
import os
import joblib
os.makedirs("model", exist_ok=True)
joblib.dump(best_model, "model/mileage_model.pkl")
print("Model saved!")


Model saved!


In [212]:
X_train.head(1)

,Engine,Kerb Weight,Transmission Type,Power,No. of Cylinders,Registration Year,Fuel_Diesel,Fuel_Petrol
336,1197.0,1100.0,0.0,81.86,4.0,2018.0,False,True
